To be able to run this baseline, it is necessary to add the shroom-vis-images folder inside this one, and then it is possible to run the notebook. The results of the grid search are saved in a CSV file, which can be used to analyze the performance of the different models and hyperparameters.

In [ ]:
from scripts.llava import llava_loading, llava_features_from_response
import os
from tqdm.auto import tqdm

from scripts.train_hallushift_models import (
    features_from_df,
    TokenDataset,
    MLP,
    evaluate,
    random_baseline,
    NONE_LABEL,
)

In [ ]:
import ast
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
HIDDEN_DIR = PROJECT_ROOT / "hidden"
SOURCE_DIR = PROJECT_ROOT / "notebook_jsonl_sources"

HIDDEN_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
folder_path = "data"
data = []

for filename in os.listdir(folder_path):
    if filename.endswith("labeled.jsonl"):
        file_path = os.path.join(folder_path, filename)

        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    data.append(json.loads(line))

df = pd.DataFrame(data)

df.head()

In [ ]:
def parse_labels(x):
    if isinstance(x, list):
        return x

    if pd.isna(x):
        return []

    text = str(x).strip()

    if text == "" or text == "[]":
        return []

    try:
        return ast.literal_eval(text)
    except Exception:
        try:
            return json.loads(text)
        except Exception:
            return []


df["labels_parsed"] = df["labels"].apply(parse_labels)
df

In [ ]:
jsonl_sources = {}

for (language, split), group in df.groupby(["language", "split"]):
    language = str(language)
    split = str(split)

    out_path = SOURCE_DIR / f"notebook_{language}_{split}.jsonl"

    with out_path.open("w", encoding="utf-8") as f:
        for _, row in group.iterrows():
            obj = {
                "id": str(row["id"]),
                "image": Path(str(row["image_name"])).name,
                "prompt": str(row["prompt"]),
                "response": str(row["response"]),
                "labels": row["labels_parsed"],
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

    jsonl_sources[(language, split)] = out_path.relative_to(PROJECT_ROOT)

jsonl_sources

In [ ]:
IMAGE_DIR = Path("shroom-vis-images")


def build_image_index(image_dir: Path):
    image_index = {}

    for p in image_dir.rglob("*"):
        if p.is_file():
            image_index[p.name] = p

    return image_index


image_index = build_image_index(IMAGE_DIR)

print(f"Indexed {len(image_index)} images")

In [ ]:
def find_image_path(image_name: str, image_dir: Path, image_index: dict):
    image_name = str(image_name).strip()

    direct = image_dir / image_name
    if direct.exists():
        return direct

    base = Path(image_name).name
    if base in image_index:
        return image_index[base]

    normalized_target = base.lower().replace(" ", "_")

    for fname, path in image_index.items():
        if fname.lower().replace(" ", "_") == normalized_target:
            return path

    return None

In [ ]:
model, processor = llava_loading()
tokenizer = processor.tokenizer

print("Loaded LLaVA")

In [ ]:
def summarize_generated_tokens(generated_obj, tokenizer, response_text):
    import torch
    import torch.nn.functional as F

    token_summaries = []

    logits_tuple = getattr(generated_obj, "logits", None) or []
    hidden_list = getattr(generated_obj, "hidden_states", None) or []
    attn_list = getattr(generated_obj, "attentions", None) or []

    response_ids = tokenizer.encode(response_text, add_special_tokens=False)
    response_len = len(response_ids)

    for t in range(response_len):
        summary = {"token_index": t}

        token_id = int(response_ids[t])
        summary["token_id"] = token_id

        try:
            summary["raw_token"] = tokenizer.convert_ids_to_tokens(token_id)
        except Exception:
            summary["raw_token"] = None

        try:
            summary["token_text"] = tokenizer.decode([token_id])
        except Exception:
            summary["token_text"] = ""

        try:
            logit = logits_tuple[t] if t < len(logits_tuple) else None

            if logit is not None:
                probs = F.softmax(logit[0].to(torch.float32), dim=0)

                max_prob = float(probs.max().item())
                target_prob = float(probs[token_id].item()) if token_id < probs.shape[-1] else None

                summary["max_prob"] = max_prob
                summary["target_prob"] = target_prob

                if target_prob is not None:
                    summary["token_nll"] = float(
                        -torch.log(torch.tensor(target_prob + 1e-12)).item()
                    )
                    summary["token_perplexity"] = float(1.0 / max(target_prob, 1e-12))
                else:
                    summary["token_nll"] = None
                    summary["token_perplexity"] = None
            else:
                summary["max_prob"] = None
                summary["target_prob"] = None
                summary["token_nll"] = None
                summary["token_perplexity"] = None

        except Exception:
            summary["max_prob"] = None
            summary["target_prob"] = None
            summary["token_nll"] = None
            summary["token_perplexity"] = None

        # Hidden-state norms
        hidden_norms = []

        try:
            if t < len(hidden_list):
                per_layer = hidden_list[t]

                for layer_tensor in per_layer:
                    try:
                        if layer_tensor is not None and hasattr(layer_tensor, "dim"):
                            val = float(layer_tensor[0].to(torch.float32).norm().item())
                        elif layer_tensor is not None:
                            val = float(torch.tensor(layer_tensor).to(torch.float32).norm().item())
                        else:
                            val = 0.0
                    except Exception:
                        val = 0.0

                    hidden_norms.append(val)

        except Exception:
            hidden_norms = []

        summary["hidden_layer_norms"] = hidden_norms

        # Attention summaries
        attn_means = []
        attn_entropies = []

        try:
            if t < len(attn_list):
                per_attn_layers = attn_list[t]

                if per_attn_layers is not None:
                    for att_layer in per_attn_layers:
                        try:
                            if (
                                att_layer is not None
                                and hasattr(att_layer, "dim")
                                and att_layer.dim() == 4
                            ):
                                # Common shape: batch, heads, query_len, key_len
                                att = att_layer[0].to(torch.float32)

                                # Average over heads and query positions
                                att_vec = att.mean(dim=0).mean(dim=0)

                                att_mean = float(att_vec.mean().item())

                                probs = att_vec / (att_vec.sum() + 1e-12)
                                entropy = -float(
                                    (probs * torch.log(probs + 1e-12)).sum().item()
                                )
                            else:
                                att_mean = (
                                    float(att_layer.mean().item())
                                    if hasattr(att_layer, "mean")
                                    else 0.0
                                )
                                entropy = 0.0

                        except Exception:
                            att_mean = 0.0
                            entropy = 0.0

                        attn_means.append(att_mean)
                        attn_entropies.append(entropy)

        except Exception:
            attn_means = []
            attn_entropies = []

        summary["attn_means"] = attn_means
        summary["attn_entropies"] = attn_entropies

        token_summaries.append(summary)

    return token_summaries

In [ ]:
def token_char_spans_from_text(tokenizer, text):
    token_ids = tokenizer.encode(text, add_special_tokens=False)

    spans = []
    cursor = 0

    for token_id in token_ids:
        token_text = tokenizer.decode([token_id])
        clean_token = token_text.strip()

        if clean_token:
            pos = text.find(clean_token, cursor)
        else:
            pos = cursor

        if pos == -1:
            start = cursor
            end = cursor
        else:
            start = pos
            end = pos + len(clean_token)
            cursor = end

        spans.append(
            {
                "token_id": int(token_id),
                "token_text": token_text,
                "char_start": start,
                "char_end": end,
            }
        )

    return spans

def assign_token_labels(token_summaries, response_text, labels, tokenizer):
    spans = token_char_spans_from_text(tokenizer, response_text)

    for i, tok in enumerate(token_summaries):
        if i < len(spans):
            tok_start = spans[i]["char_start"]
            tok_end = spans[i]["char_end"]

            tok["char_start"] = tok_start
            tok["char_end"] = tok_end
        else:
            tok_start = None
            tok_end = None

            tok["char_start"] = None
            tok["char_end"] = None

        matched = []

        for lab in labels:
            lab_start = lab.get("start")
            lab_end = lab.get("end", len(response_text))

            if lab_start is None:
                continue

            if tok_start is None or tok_end is None:
                continue

            overlaps = tok_start < lab_end and tok_end > lab_start

            if overlaps:
                matched.append(lab)

        tok["hallucination_labels"] = matched
        tok["is_labeled_hallucination"] = len(matched) > 0

    return token_summaries

In [ ]:
def process_dataframe_with_llava(
    df,
    image_dir,
    image_index,
    model,
    processor,
    tokenizer,
    max_rows=None,
    include_label_alignment=True,
):
    all_samples = []
    missing_images = []
    errors = []

    rows = df if max_rows is None else df.head(max_rows)

    for _, row in tqdm(rows.iterrows(), total=len(rows)):
        sample_id = row["id"]
        split = row["split"]
        language = row["language"]
        prompt = row["prompt"]
        image_name = row["image_name"]
        response = row["response"]
        labels = row["labels_parsed"]

        image_path = find_image_path(image_name, image_dir, image_index)

        if image_path is None:
            missing_images.append(
                {
                    "id": sample_id,
                    "image_name": image_name,
                }
            )
            continue

        try:
            res = llava_features_from_response(
                model=model,
                processor=processor,
                prompt=prompt,
                image_path=str(image_path),
                response_text=response,
                include_features=False,
            )

            if res is None:
                errors.append(
                    {
                        "id": sample_id,
                        "image_name": image_name,
                        "error": "llava_features_from_response returned None",
                    }
                )
                continue

            generated_obj = res.get("generated", None)

            if generated_obj is None:
                errors.append(
                    {
                        "id": sample_id,
                        "image_name": image_name,
                        "error": "missing generated object",
                    }
                )
                continue

            token_summaries = summarize_generated_tokens(
                generated_obj=generated_obj,
                tokenizer=tokenizer,
                response_text=response,
            )

            if include_label_alignment:
                token_summaries = assign_token_labels(
                    token_summaries=token_summaries,
                    response_text=response,
                    labels=labels,
                    tokenizer=tokenizer,
                )

            sample = {
                "meta": {
                    "id": sample_id,
                    "split": split,
                    "language": language,
                    "image": image_name,
                    "image_path": str(image_path),
                    "prompt": prompt,
                    "response": response,
                    "labels": labels,
                    "model_backend": "llava",
                },
                "tokens": token_summaries,
            }

            all_samples.append(sample)

        except Exception as e:
            err = str(e)

            errors.append(
                {
                    "id": sample_id,
                    "image_name": image_name,
                    "error": err,
                }
            )
            continue

    return all_samples, missing_images, errors

In [ ]:
samples, missing_images, errors = process_dataframe_with_llava(
    df=df,
    image_dir=IMAGE_DIR,
    image_index=image_index,
    model=model,
    processor=processor,
    tokenizer=tokenizer,
    max_rows=None,
    include_label_alignment=True,
)

print(f"Extracted samples: {len(samples)}")
print(f"Missing images: {len(missing_images)}")
print(f"Errors: {len(errors)}")

In [ ]:
SMOKE_TEST = False
if SMOKE_TEST:
    FEATURE_PATH = Path("small_samples_per_token_llava.json")
else:
    FEATURE_PATH = Path("all_samples_per_token_llava.json")

with FEATURE_PATH.open("r", encoding="utf-8") as f:
    payload = json.load(f)

samples = payload["samples"]

print(f"Loaded {len(samples)} samples")

In [ ]:
from collections import defaultdict

samples_by_lang_split = defaultdict(list)

for sample in samples:
    meta = sample["meta"]

    language = str(meta["language"])
    split = str(meta["split"])

    samples_by_lang_split[(language, split)].append(sample)

samples_by_lang_split.keys()

In [ ]:
for (language, split), group_samples in samples_by_lang_split.items():
    out_dir = HIDDEN_DIR / f"per_token_hidden_{language}_train_llava"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"all_samples_per_token_llava_{language}_train.json"

    payload_out = {
        "metadata": {
            "total_samples": len(group_samples),
            "jsonl_source": str(jsonl_sources[(language, split)]),
            "model_backend": "llava",
            "language": language,
            "split": split,
        },
        "samples": group_samples,
    }

    with out_path.open("w", encoding="utf-8") as f:
        json.dump(payload_out, f, indent=2, ensure_ascii=False)

    print("Saved:", out_path)

In [ ]:
FEATURE_PATH = Path("all_samples_per_token_llava_test.json")

with FEATURE_PATH.open("r", encoding="utf-8") as f:
    payload = json.load(f)

samples = payload["samples"]

print(f"Loaded {len(samples)} samples")

In [ ]:
from collections import defaultdict

samples_by_lang_split = defaultdict(list)

for sample in samples:
    meta = sample["meta"]

    language = str(meta["language"])
    split = str(meta["split"])

    samples_by_lang_split[(language, split)].append(sample)

samples_by_lang_split.keys()

In [ ]:
for (language, split), group_samples in samples_by_lang_split.items():
    out_dir = HIDDEN_DIR / f"per_token_hidden_{language}_test_llava"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"all_samples_per_token_llava_{language}_test.json"

    payload_out = {
        "metadata": {
            "total_samples": len(group_samples),
            "jsonl_source": str(jsonl_sources[(language, split)]),
            "model_backend": "llava",
            "language": language,
            "split": split,
        },
        "samples": group_samples,
    }

    with out_path.open("w", encoding="utf-8") as f:
        json.dump(payload_out, f, indent=2, ensure_ascii=False)

    print("Saved:", out_path)

In [ ]:
df_results = pd.read_csv(HIDDEN_DIR / "hallushift_predictions_llava_binary.csv")
df_results.head()

In [ ]:
grid = [
    {"classifier": "mlp", "epochs": 10, "batch_size": 32, "lr": 1e-4, "weight_decay": 0.0},
    {"classifier": "mlp", "epochs": 10, "batch_size": 64, "lr": 3e-4, "weight_decay": 1e-4},

    {"classifier": "random_forest", "n_estimators": 300, "max_depth": None, "min_samples_leaf": 1},
    {"classifier": "random_forest", "n_estimators": 500, "max_depth": 20, "min_samples_leaf": 2},

    {"classifier": "logistic_regression", "C": 0.1},
    {"classifier": "logistic_regression", "C": 1.0},
    {"classifier": "logistic_regression", "C": 10.0}
]

In [ ]:
import torch
from scripts.train_hallushift_models import train_one_model

all_results = []

for cfg in grid:
    print("\n==============================")
    print("Running config:", cfg)

    res = train_one_model(
        project_root=PROJECT_ROOT,
        hidden_dir=HIDDEN_DIR,
        model_name="llava",
        train_on_redacted=False,
        label_mode="multiclass",
        epochs=10,
        batch_size=32,
        lr=1e-4,
        val_sentence_ratio=0.1,
        seed=42,
        device="cuda" if torch.cuda.is_available() else "cpu",
        quick_mode=False,
        model_config=cfg,
    )

    all_results.append(res)

results_df = pd.DataFrame(all_results)
results_df.sort_values("best_val_f1", ascending=False)

In [ ]:
results_df.to_csv("hallushift_model_grid_search_results_full.csv", index=False)

### This part of the code is needed to translate the output of the method in the same format requested by the evaluation script

In [ ]:
def parse_ref_file(fname):
    df_ref = pd.read_json(fname, lines=True)
    lang = fname.name.split('.')[2]
    df_ref['sid'] = range(len(df_ref))
    df_ref['sentence_id'] = [f'{lang}:test:{idx}' for idx in range(len(df_ref))]
    df_ref['metadata'] = df_ref.apply(lambda row: {'strategy': row['strategy'], **row['metadata']}, axis=1)
    df_ref['actual_id'] = df_ref['id']
    return df_ref[['sid', 'sentence_id', 'language', 'image_name', 'metadata', 'response', 'actual_id']]

df_ref = pd.concat(map(parse_ref_file, Path('data/hidden').glob('*test*')))
df_ref['offset_mapping'] = df_ref['response'].apply(lambda response: tokenizer.tokenizer(response, return_offsets_mapping=True).offset_mapping[1:])

df = pd.read_csv(f'submission.csv')

df['image_name'] = df.sentence_id.apply(lambda sid: ':'.join(sid.split(':')[3:]))
df['sid'] = df.sentence_id.apply(lambda sid: int(sid.split(':')[2]))


total_sid_offsets_allowed = len(df_ref) - len(df.sentence_id.unique())
sid_remaps = {}
for lang, subdf in df.groupby('language'):
    sid_offset = 0
    for sid in subdf.sid.unique():
        image = subdf[subdf.sid == sid].image_name.unique()
        assert len(image) == 1
        image = image[0]
        ref_image = df_ref[(df_ref.sid == sid + sid_offset) & (df_ref.language == lang)].image_name
        assert len(ref_image.unique()) == 1
        ref_image = ref_image.iloc[0]
        found = ref_image == image
        while not found:
            sid_offset += 1
            total_sid_offsets_allowed -= 1
            ref_image = df_ref[(df_ref.sid == sid + sid_offset) & (df_ref.language == lang)].image_name
            assert len(ref_image.unique()) == 1
            ref_image = ref_image.iloc[0]
            found = ref_image == image
            assert total_sid_offsets_allowed >= 0, f'{lang}:test:{sid}'
        sid_remaps[f'{lang}:test:{sid}:{image}'] = f'{lang}:test:{sid + sid_offset}'
    
df['sentence_id_'] = df['sentence_id']
df['sentence_id'] = df['sentence_id'].map(sid_remaps)
df['sid_'] = df['sid']
del df['sid']

df = df.merge(df_ref)

def token_span(row):
    mapping = row["offset_mapping"]
    token_index = row["token_index"]
    if token_index < len(mapping):
        return mapping[token_index]
    token_text = str(row.get("token_text", "")).strip("\u2581")
    end = len(row["response"])
    start = max(0, end - len(token_text)) if token_text else end
    return (start, end)

df[["token_start_index", "token_end_index"]] = df.apply(token_span, axis=1, result_type="expand")


df['strategy'] = df['metadata'].apply(lambda md: md['strategy'])

df['origin'] = df['metadata'].apply(lambda md: md.get('model', ' ' +md['strategy']))

for lang, df_ in df.groupby('language'):
    records = []
    for sentence_id, subdf in df_.groupby('sentence_id'):
        assert len(subdf.actual_id.unique()) == 1
        record = {'id': subdf.actual_id.iloc[0], 'labels': []}
        for _, row in subdf[subdf['pred_label'] != 'none'].iterrows():
            span = {
                'start': row['token_start_index'],
                'end' : row['token_end_index'],
                'label': 'invention', #'row['pred_label'],
                'prob': row['pred_prob'],
            }
            record['labels'].append(span)
        records.append(record)
    pd.DataFrame.from_records(records).to_json(f'final_submission.{lang}.jsonl', lines=True, orient='records')